# Tool 3bis: Automatic sleep scoring (GSSC, YASA, SleepFM)

Scores every 30-s epoch of a night automatically with up to **three published models**, in two steps:

1. **Automatic scoring** (Sections 1-3). Each model you tick scores each night. Its hypnogram is written beside
   the EDF (tool-3 format, so tools 5-9 can use it in place of an expert one), with the **probability of each
   stage** for every epoch (the **hypnodensity**). Where an expert hypnogram exists, each model is compared with
   it night by night (confusion matrix, Cohen's kappa, agreement per stage).
2. **Comparison of the models** (Section 4). The models side by side against the expert hypnogram, on every
   night together: agreement table, confusion matrices, hypnograms, hypnodensities, and two **recalibrations of
   SleepFM** learnt on your own expert hypnograms.

A night where the probabilities are clear-cut is easy to score; a night with many uncertain epochs is ambiguous
(for the machine and often for the scorer too).

### The three models in a few words

| | What it is | Trained on | Needs | Works best on |
|---|---|---|---|---|
| **GSSC** | a deep neural network (Greifswald, 2023) | ~3,300 clinical nights of adults and children (NSRR) | 1 EEG and/or the EOG | clinical PSG; robust with few channels, even the EOG alone |
| **YASA** | decision trees on hand-made EEG measures (2021) | ~3,000 nights (NSRR), all ages and health conditions | 1 central EEG; EOG and chin EMG improve it | whole nights with a central EEG; fast and light |
| **SleepFM** | a "foundation model" of the whole PSG (Stanford, 2026) | over 585,000 hours of PSG, ~65,000 people | EEG/EOG, chin EMG, ECG and respiration | complete PSG montages (sleep staging is a side task of this model) |

All three were trained on standard clinical montages, where each EEG electrode is referenced to the mastoid of
the **other** side (C3-M2, C4-M1...). That is the reference given to them by default (Section 2), whatever
reference tool 2 declared for the other tools.

<details>
<summary><b>Details for experts: training, inputs, preprocessing, performance</b> (click to open)</summary>

| | **YASA** | **GSSC** | **SleepFM-Clinical** |
|---|---|---|---|
| Reference | Vallat & Walker 2021, *eLife* | Hanna & Flöel 2023, *Front. Neuroinform.* | Thapa et al. 2026, *Nature Medicine* |
| Model | gradient-boosted trees (LightGBM) on hand-crafted features | 1-D ResNets (one EEG branch, one EOG branch) + bidirectional GRU (context of the night) | transformer pretrained by contrastive learning, then a sleep-staging head (transformer + bidirectional LSTM) |
| Training data | ~3,000 NSRR nights (varied ages, sexes, pathologies) | 3,314 NSRR nights: SHHS2 (936), CFS (730), NCHSDB (665), WSC (983); external test DREEM (80 subjects, consensus of 5 scorers) | pretraining: > 585,000 h of PSG, ~65,000 participants (Stanford Sleep Dataset released; SHHS left out) |
| EEG | **1 central, required**, on the mastoid (C4-M1 preferred, C3-M2) | C3, C4, F3, F4 on the contralateral mastoid (or Fpz); 1 or 2 EEG recommended | 0 to 10 "BAS" channels (EEG + EOG) |
| EOG | optional: left EOG (LOC/E1-M2) preferred | **HEOG = left EOG - right EOG** (left or right alone also works; VEOG not recommended) | part of BAS |
| EMG | optional: chin | not used | EMG modality: 1 to 4 channels |
| Other | age/sex optional (not used here) | none | **RESP** (1 to 7 channels: thorax, abdomen, flow, SpO2, pulse) and **EKG** (1 to 2) |
| Internal preprocessing | resampled to 100 Hz, band-pass 0.4-30 Hz, robust z-score over the night. **Do not filter before** | band-pass 0.3-30 Hz, z-score per channel and night, 85.33 Hz (2,560 samples per 30 s) | anti-alias low-pass, interpolation to 128 Hz, z-score per channel and night |
| Output | one stage per 30 s | one stage per 30 s | **one prediction per 5 s** (averaged over each 30-s epoch here) |
| Published performance | ~85 % agreement with the expert consensus; N1 the weakest | > 80 % agreement on every test set, including one EEG channel or the EOG alone | mean F1 0.70-0.78 for sleep staging (a secondary task: the authors say a specialised model would do better) |
| Licence | BSD-3 | **AGPL-3.0** | **CC BY-NC 4.0** (non-commercial use only) |

- **YASA** computes, per 30-s epoch and per channel: standard deviation, interquartile range, skewness,
  kurtosis, zero crossings, Hjorth parameters, absolute and relative band powers, power ratios, permutation
  entropy and fractal dimensions. It smooths them over 7.5 min (centred) and 2 min (past), then normalises them
  over the whole night: it needs **the whole night**, not pieces. Sampling rate above 80 Hz.
- **GSSC** uses a "permutation consensus": it scores the night with **every combination** of the signals it is
  given (each EEG with the EOG, each EEG alone, the EOG alone) and keeps, epoch by epoch, the most confident
  combination. With 4 EEG and 1 EOG that is 9 passes over the night.
- **SleepFM** cuts each channel into 5-s tokens at 128 Hz; an attention layer pools the channels of a modality
  (hence any number of channels), a transformer works on 5-min chunks and the staging head (bidirectional LSTM)
  reads the whole night (up to 11.4 h). It needs **at least one channel in each of its 4 modalities** (BAS,
  RESP, EKG, EMG). The signals are band-passed first (EEG/EOG 0.3-35 Hz, chin EMG 10-100 Hz): without it 50-Hz
  mains passes its filters and the model gives almost no REM.
- **SleepFM recalibration A** learns a "temperature" and one bias per stage (Wake is the reference) on the
  other nights, by leave-one-night-out: at least 2 nights. **Recalibration B** refits SleepFM's final layer on
  its frozen internal features, its penalty chosen by an inner cross-validation: at least 3 nights (6 or more
  advised). Both learn from **your expert hypnograms** (a supervised adaptation); each night is predicted with
  parameters learnt without it, so the comparison stays fair.
</details>

**Inputs**: the EDF files, tool 2's `config_param/remap_reref_persubject.json` (channel names, and the EOG,
chin EMG and ECG declared as context channels in tool 2's Section 2bis) and, optionally, the expert hypnogram
remapped by tool 3 (`{id}_Hypnogram_remapped.txt`). SleepFM also takes the respiratory channels, found by name
(Thoracic, Abdominal, Flux/Airflow, SpO2, Pulse).

**Environment**: GSSC and YASA run in the `inspect_edf` environment. SleepFM needs the autoscoring environment
(`autoscore`); its model files (~29 MB, official repository) are downloaded automatically the first time.

**Outputs** (`<algo>` = `gssc`, `yasa` or `sleepfm`):

```
{data_folder}/
  {id}_hypnogram_<algo>.txt               ← automatic hypnogram beside the EDF (tool-3 format: one label per line)
  derivatives/autoscoring_<algo>/<subtree>/
    {id}_hypnodensity.tsv                 ← 1 row/epoch: stage, p_W … p_R, confidence, expert stage
    {id}_scoring_params.json              ← channels, reference, versions (provenance)
    {id}_hypnodensity_5s.tsv              ← SleepFM only: its native 5-s probabilities
    {id}_sleepfm_features.npz             ← SleepFM only: its internal features, for recalibration B
  reports_autoscoring_<algo>/<subtree>/
    {id}_autoscoring_comparison.tsv       ← agreement metrics with the expert hypnogram (1 row)
    {id}_confusion.tsv                    ← confusion matrix, long format (expert, <algo>, n)
    {id}_autoscoring_report.html          ← hypnograms, hypnodensity, confusion matrix
  reports_autoscoring_<algo>/
    global_autoscoring_summary.tsv        ← every participant (rebuilt from the per-file tables on disk)
    global_confusion.tsv                  ← pooled confusion matrix
    global_autoscoring_report.html
  autoscore_outputs/                      ← Section 4: the models compared (same files as Score_EDF)
```

Work through the sections in order: **1** pick the study folder and scan, **2** choose the models and the
channels, **3** run, **4** compare the models.

In [ ]:
try:
    import os
    import json
    import re
    import time
    import datetime
    import warnings
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    import mne
    from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, confusion_matrix

    import ipywidgets as widgets
    from IPython.display import display, HTML
    from ipyfilechooser import FileChooser
except ImportError as e:
    print("⚠️ Import error, a package is missing:", e)
else:
    print("✅ Packages imported successfully!")

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

# Shared libraries (tools/participant_selection_lib.py, tools/autoscoring_lib.py), probed from the repo root (the
# canonical launch folder) or from tools/, so the import works whatever folder Voila is launched from.
import sys
for _p in [os.getcwd(), os.path.join(os.getcwd(), 'tools'),
           os.path.join(os.path.dirname(os.getcwd()), 'tools'), os.path.join(os.getcwd(), '..', 'tools')]:
    if os.path.isfile(os.path.join(_p, 'participant_selection_lib.py')):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        break
import participant_selection_lib as PSL

In [ ]:
# The computation and the figures of this tool live in shared libraries: tools/autoscoring_lib.py (step 1,
# shared with tool 0), tools/sleepfm_lib.py (SleepFM) and tools/autoscoring_compare_lib.py (step 2, Section 4).
from autoscoring_lib import (
    GSSC_OK, GSSC_MSG, GSSC_VERSION, TORCH_VERSION, CONSENSUS_RULE, EPOCH_SEC, STAGES,
    HYPNODENSITY_ROWS_PER_FIG, clock_hours, compare_scoring, eeg_options, list_edf,
    load_custom_stages, load_expert, load_json_lenient, metrics_html, non_aasm_profile, params_html,
    plot_confusion, plot_hypnodensity, plot_hypnodensity_stack, plot_hypnograms, plot_non_aasm_profile,
    pool_profiles, recording_start, save_report_html,
    ALGORITHMS, EOG_MODES, REFERENCE_OPTIONS, REFERENCE_CONTRALATERAL, YASA_OK, YASA_MSG, YASA_VERSION,
    non_aasm_help, score_night_gssc, score_night_sleepfm, score_night_yasa, YASA_EMG_DECLARED)
from edf_io_lib import read_edf_header_custom
import sleepfm_lib as SFM
import autoscoring_compare_lib as ACL
if GSSC_OK:
    import torch
    from gssc.infer import EEGInfer
    print(f"✅ GSSC {GSSC_VERSION} (torch {TORCH_VERSION}) ready"
          f"{', GPU available' if torch.cuda.is_available() else ', running on CPU'}.")
else:
    display(HTML(GSSC_MSG))
if YASA_OK:
    print(f"✅ YASA {YASA_VERSION} ready.")
else:
    display(HTML(f'<div style="color:#c0392b;">{YASA_MSG}</div>'))
if SFM.SLEEPFM_DEPS_OK:
    print("✅ SleepFM can run (its model is checked in Section 2).")
else:
    display(HTML(f'<div style="color:#e67e22;">⚠ {SFM.SLEEPFM_DEPS_MSG}</div>'))

## 1. Study folder

Select the folder holding the EDF files (searched recursively), then click **Scan**. Tool 2's
`config_param/remap_reref_persubject.json` is read from this folder. The **expert hypnogram suffix** is
detected automatically (leave it empty when there is no expert hypnogram: the nights are then scored
without comparison). The **output suffixes** name the automatic hypnogram each model writes beside each EDF.

In [ ]:
fc_data = FileChooser()
fc_data.title = '<b>Select your study folder:</b>'
fc_data.show_only_dirs = True

hypno_suffix = widgets.Text(value='_Hypnogram_remapped.txt', description='Expert hypno suffix:',
                            style={'description_width': '150px'}, layout=widgets.Layout(width='460px'))
hypno_suffix_info = widgets.HTML(value='')
# One output suffix per model: the automatic hypnogram each one writes beside the EDF.
out_suffix = widgets.Text(value='_hypnogram_gssc.txt', description='GSSC output suffix:',
                          style={'description_width': '150px'}, layout=widgets.Layout(width='460px'))
out_suffix_yasa = widgets.Text(value='_hypnogram_yasa.txt', description='YASA output suffix:',
                               style={'description_width': '150px'}, layout=widgets.Layout(width='460px'))
out_suffix_sleepfm = widgets.Text(value='_hypnogram_sleepfm.txt', description='SleepFM output suffix:',
                                  style={'description_width': '150px'}, layout=widgets.Layout(width='460px'))
OUT_SUFFIX = {'GSSC': out_suffix, 'YASA': out_suffix_yasa, 'SleepFM': out_suffix_sleepfm}
ALGO_KEY = {'GSSC': 'gssc', 'YASA': 'yasa', 'SleepFM': 'sleepfm'}   # folder and column names
btn_scan = widgets.Button(description='Scan', button_style='info', icon='search',
                          layout=widgets.Layout(width='200px', height='34px'))
scan_info = widgets.HTML(value='')
participant_selector = PSL.ParticipantSelector()

STATE = {'cfg': {}, 'edf_by_id': {}, 'not_in_cfg': set(), 'custom': []}


def _detect_hypno_suffix(chooser):
    """Auto-fill the expert hypnogram suffix: among the .txt suffixes found for >= 50 % of the most
    frequent one, prefer the longest (the remapped version). Event exports and the models' outputs are skipped."""
    try:
        if not chooser.selected:
            hypno_suffix_info.value = ''
            return
        folder = Path(chooser.selected)
        edf_files = list_edf(folder)
        if not edf_files:
            hypno_suffix_info.value = '<small style="color:#888;">No EDF file found (recursive scan).</small>'
            return
        all_txt = [f for f in folder.rglob('*') if f.suffix.lower() == '.txt']
        counts = {}
        for edf in edf_files:
            for t in all_txt:
                if os.path.normcase(t.name).startswith(os.path.normcase(edf.stem)):
                    suf = t.name[len(edf.stem):]
                    counts[suf] = counts.get(suf, 0) + 1
        outputs = {os.path.normcase(w.value) for w in OUT_SUFFIX.values()}
        sel = {s: c for s, c in counts.items()
               if 'event' not in s.lower() and 'summary' not in s.lower() and 'studylog' not in s.lower()
               and os.path.normcase(s) not in outputs}
        if not sel:
            hypno_suffix.value = ''
            hypno_suffix_info.value = ('<small style="color:#e67e00;">No expert hypnogram detected: the '
                                       'nights will be scored without comparison.</small>')
            return
        max_count = max(sel.values())
        cands = {s: c for s, c in sel.items() if c >= max_count * 0.5}
        best, best_n = max(cands.items(), key=lambda x: (len(x[0]), x[1]))
        hypno_suffix.value = best
        parts = [f'<b>{s}</b>&nbsp;(×{c}){"&nbsp;← selected" if s == best else ""}'
                 for s, c in sorted(counts.items(), key=lambda x: -x[1])]
        color = '#2e7d32' if best_n == len(edf_files) else '#e67e00'
        hypno_suffix_info.value = (f'<small style="color:{color};">Detected: &nbsp;{"&nbsp;·&nbsp;".join(parts)}'
                                   f'&nbsp;— {best_n}/{len(edf_files)} files matching</small>')
    except Exception as e:
        hypno_suffix_info.value = f'<small style="color:#c0392b;">Error detecting hypnograms: {e}</small>'


fc_data.register_callback(_detect_hypno_suffix)


def output_dirs(data_root, edf_path, algo='GSSC'):
    rel = edf_path.parent.relative_to(data_root)
    key = ALGO_KEY[algo]
    return (data_root / 'derivatives' / f'autoscoring_{key}' / rel,
            data_root / f'reports_autoscoring_{key}' / rel)


def output_paths(data_root, edf_path, fid, algo='GSSC'):
    deriv, reports = output_dirs(data_root, edf_path, algo)
    paths = {'hypno': edf_path.parent / f'{fid}{OUT_SUFFIX[algo].value}',
             'density': deriv / f'{fid}_hypnodensity.tsv',
             'params': deriv / f'{fid}_scoring_params.json',
             'comparison': reports / f'{fid}_autoscoring_comparison.tsv',
             'confusion': reports / f'{fid}_confusion.tsv',
             'report': reports / f'{fid}_autoscoring_report.html'}
    if algo == 'SleepFM':
        paths['density_5s'] = deriv / f'{fid}_hypnodensity_5s.tsv'      # native 5-s probabilities
        paths['features'] = deriv / f'{fid}_sleepfm_features.npz'       # for recalibration B (Section 4)
    return paths


def algo_outputs(data_root, edf_path, fid, algo):
    """(complete, partial) for one model's outputs: complete = automatic hypnogram + hypnodensity (+ SleepFM's
    5-s table and features) + report all on disk (the data are written before the report)."""
    p = output_paths(data_root, edf_path, fid, algo)
    keys = ['hypno', 'density', 'report'] + (['density_5s', 'features'] if algo == 'SleepFM' else [])
    have = [p[k].exists() for k in keys]
    return all(have), any(have) and not all(have)


def refresh_done():
    """'Already processed' = every ticked model's outputs on disk (recomputed when the models change).
    Returns the participants with incomplete outputs of a ticked model (they are rescored)."""
    if not fc_data.selected or not STATE['edf_by_id']:
        return []
    data_root = Path(fc_data.selected)
    algos = ticked_algorithms()
    done, mismatch = [], []
    for fid, edf in STATE['edf_by_id'].items():
        if fid not in STATE['cfg']:
            continue
        states = [algo_outputs(data_root, edf, fid, a) for a in algos]
        if algos and all(complete for complete, _ in states):
            done.append(fid)
        elif any(partial for _, partial in states):
            mismatch.append(fid)
    participant_selector.set_participants(list(STATE['edf_by_id']), done=done)
    STATE['n_done'] = len(done)
    return mismatch


def on_scan(btn):
    btn.disabled = True
    try:
        if not fc_data.selected:
            scan_info.value = '<span style="color:#c0392b;">Select the study folder first.</span>'
            return
        data_root = Path(fc_data.selected)
        cfg_path = data_root / 'config_param' / 'remap_reref_persubject.json'
        if not cfg_path.exists():
            scan_info.value = (f'<span style="color:#c0392b;">{cfg_path} not found: run tool 2 '
                               '(2_select&amp;remap_channels) first.</span>')
            return
        try:
            cfg = load_json_lenient(cfg_path)
        except Exception as e:
            scan_info.value = f'<span style="color:#c0392b;">Cannot read {cfg_path.name}: {e}</span>'
            return
        edf_files = list_edf(data_root)
        cfg_keys = {os.path.normcase(k): k for k in cfg}
        STATE['cfg'] = {}
        STATE['edf_by_id'] = {}
        STATE['not_in_cfg'] = set()
        n_eog = 0
        options = []
        for edf in edf_files:
            fid = edf.stem
            STATE['edf_by_id'][fid] = edf
            key = cfg_keys.get(os.path.normcase(fid))
            if key is None:
                STATE['not_in_cfg'].add(fid)
                continue
            STATE['cfg'][fid] = cfg[key]
            ctx = cfg[key].get('context_channels') or {}
            if ctx.get('eog_left') or ctx.get('eog_right'):
                n_eog += 1
            for ch in eeg_options(cfg[key]):
                if ch not in options:
                    options.append(ch)
        # Every channel name of the recordings (header only): the choices of YASA's chin EMG (Section 2)
        channel_names = set()
        for edf in edf_files:
            try:
                channel_names.update(read_edf_header_custom(edf)['channel'])
            except Exception:
                pass
        set_emg_options(sorted(channel_names))
        mismatch = refresh_done()
        STATE['custom'] = load_custom_stages(data_root)
        set_eeg_options(options)
        n_cfg = len(STATE['cfg'])
        msg = (f'<span style="color:#2e7d32;">{len(edf_files)} EDF file(s): {n_cfg} configured in tool 2, '
               f'{n_eog} with an EOG declared (context channels), {STATE["n_done"]} already scored'
               f' ({", ".join(ticked_algorithms()) or "no model ticked"}).</span>')
        if STATE['custom']:
            msg += (f'<br><span style="color:#555;">Custom stage(s) from tool 3: {", ".join(STATE["custom"])} '
                    '(shown in a dedicated report section, outside the agreement metrics).</span>')
        if STATE['not_in_cfg']:
            msg += (f'<br><span style="color:#e67e22;">⚠ {len(STATE["not_in_cfg"])} EDF(s) not in the '
                    f'tool-2 JSON, will not be scored: {", ".join(sorted(STATE["not_in_cfg"]))}.</span>')
        if n_cfg and n_eog < n_cfg:
            msg += (f'<br><span style="color:#e67e22;">⚠ {n_cfg - n_eog} configured participant(s) without '
                    'EOG: scored on EEG only (declare the EOG in tool 2, Section 2bis).</span>')
        if mismatch:
            msg += (f'<br><span style="color:#e67e22;">⚠ {len(mismatch)} participant(s) with incomplete '
                    f'outputs, will be rescored: {", ".join(mismatch)}.</span>')
        scan_info.value = msg
    except Exception as e:
        scan_info.value = f'<span style="color:#c0392b;">Scan error: {e}</span>'
    finally:
        btn.disabled = False


btn_scan.on_click(on_scan)

display(fc_data, hypno_suffix, hypno_suffix_info, out_suffix, out_suffix_yasa, out_suffix_sleepfm, btn_scan,
        scan_info, widgets.HTML('<b>Participants to score:</b>'), participant_selector.widget)

## 2. Models and channels

Tick the **models** to run. Each scores every selected night on its own and writes its own outputs.

**EEG reference**: by default each EEG electrode is referenced to the mastoid of the other side (C3-M2, C4-M1,
F3-M2...), the montage the three models were trained on. It needs both mastoids (M1/M2 or A1/A2) in tool 2's
configuration; a participant without them is scored with tool 2's reference (said in the log). *As declared in
tool 2* uses the reference of the other tools instead.

**GSSC**. *Several electrodes (consensus)* (default, as Score_EDF did): GSSC scores with every ticked EEG and
the EOG, and keeps the most confident combination epoch by epoch, so a bad electrode on an epoch is outvoted
by a better one. Ticked by default: the electrodes GSSC was mostly trained on (C3, C4, F3, F4) that the
recordings have. A participant missing some of them is scored on the ones present. *One electrode (fast)*:
GSSC scores with that EEG and the EOG; a participant who does not have the electrode is **not** scored
(listed in the log): no other electrode is silently taken instead. EOG: the left minus right derivation
(default, what GSSC was mostly trained on), both sides separately, one side only, or none.

**YASA**: one EEG electrode (YASA prefers C4, else C3), one EOG (the left one is YASA's recommendation) and one
chin EMG: the one declared in tool 2 by default, or a channel picked in the list (the same name is then used
for every recording; a recording without it is scored without EMG, said in the log).

**SleepFM**: the AASM EEG derivations present (F3, F4, C3, C4, O1, O2), both EOG, the chin EMG and the ECG
declared in tool 2, and the respiratory channels found by name. Its model is downloaded the first time.

The models filter and resample the signals themselves: no preprocessing is needed. On CPU, count about 1-3 min
per night for GSSC (more in consensus mode), under 1 min for YASA, several minutes for SleepFM.

In [ ]:
# ---- Models ----
cb_gssc = widgets.Checkbox(value=GSSC_OK, description='GSSC', disabled=not GSSC_OK, indent=False,
                           layout=widgets.Layout(width='120px'))
cb_yasa = widgets.Checkbox(value=False, description='YASA', disabled=not YASA_OK, indent=False,
                           layout=widgets.Layout(width='120px'))
cb_sleepfm = widgets.Checkbox(value=False, description='SleepFM', disabled=not SFM.SLEEPFM_DEPS_OK, indent=False,
                              layout=widgets.Layout(width='120px'))
ALGO_CHECKS = {'GSSC': cb_gssc, 'YASA': cb_yasa, 'SleepFM': cb_sleepfm}


def ticked_algorithms():
    return [a for a, cb in ALGO_CHECKS.items() if cb.value]


dd_ref = widgets.Dropdown(options=REFERENCE_OPTIONS, value=REFERENCE_CONTRALATERAL, description='EEG reference:',
                          style={'description_width': '150px'}, layout=widgets.Layout(width='560px'))

# ---- GSSC ----
# Consensus by default (Score_EDF's choice): over the electrodes GSSC was mostly trained on, when present.
GSSC_TRAINED_EEG = ('C3', 'C4', 'F3', 'F4')
tb_mode = widgets.ToggleButtons(options=['One electrode (fast)', 'Several electrodes (consensus)'],
                                value='Several electrodes (consensus)', style={'button_width': '230px'})
dd_eeg = widgets.Dropdown(options=[], description='EEG electrode:', style={'description_width': '150px'},
                          layout=widgets.Layout(width='360px',
                                                display='' if tb_mode.value.startswith('One') else 'none'))
# Consensus mode: one checkbox per electrode (filled by the scan), plus tick-all / untick-all buttons.
eeg_checks = {}                                   # electrode name -> Checkbox, in the scan's order
box_eeg_checks = widgets.HBox([], layout=widgets.Layout(flex_flow='row wrap', margin='0 0 0 12px'))
btn_eeg_all = widgets.Button(description='Tick all', layout=widgets.Layout(width='110px'))
btn_eeg_none = widgets.Button(description='Untick all', layout=widgets.Layout(width='110px'))
consensus_hint = widgets.HTML('')
box_consensus = widgets.VBox(
    [widgets.HTML('<b>EEG electrodes:</b>'), box_eeg_checks, widgets.HBox([btn_eeg_all, btn_eeg_none]),
     consensus_hint],
    layout=widgets.Layout(display='none' if tb_mode.value.startswith('One') else ''))


def selected_eeg():
    """Electrodes ticked for the consensus mode, in the scan's order."""
    return [ch for ch, cb in eeg_checks.items() if cb.value]


def _tick_all(value):
    for cb in eeg_checks.values():
        cb.value = value


btn_eeg_all.on_click(lambda _: _tick_all(True))
btn_eeg_none.on_click(lambda _: _tick_all(False))
dd_eog = widgets.Dropdown(options=EOG_MODES, value='L − R bipolar',
                          description='EOG:', style={'description_width': '150px'},
                          layout=widgets.Layout(width='360px'))
eeg_hint = widgets.HTML('<small style="color:#888;">Run the scan (Section 1) to list the electrodes.</small>')


def _update_consensus_hint(change=None):
    """Why these electrodes are ticked, and what each extra one costs: GSSC scores the whole night once per
    combination (each EEG or none x each EOG or none, minus 'nothing')."""
    n = len(selected_eeg())
    eog_choices = {'None': 1, 'L and R separately': 3}.get(dd_eog.value, 2)
    passes = (n + 1) * eog_choices - 1
    one = 2 * eog_choices - 1                       # one electrode + the EOG
    color = '#e67e22' if n > 4 else '#555'
    consensus_hint.value = (
        '<small style="color:#555;">Ticked by default: the electrodes GSSC was mostly trained on (C3, C4, F3, '
        'F4) found in the recordings. Every extra electrode adds passes over the whole night: tick only a few '
        '(2 to 4) to keep the computing time reasonable.</small><br>'
        f'<small style="color:{color};">{n} electrode(s) ticked: {passes} combination(s), i.e. {passes} passes '
        f'over each night (about {passes / one:.0f}x the one-electrode mode).</small>')


dd_eog.observe(_update_consensus_hint, names='value')
box_gssc = widgets.VBox([widgets.HTML('<b>GSSC</b>'), tb_mode, dd_eeg, box_consensus, eeg_hint, dd_eog],
                        layout=widgets.Layout(display='' if cb_gssc.value else 'none', margin='6px 0'))

# ---- YASA ----
dd_yasa_eeg = widgets.Dropdown(options=[], description='EEG electrode:', style={'description_width': '150px'},
                               layout=widgets.Layout(width='360px'))
dd_yasa_eog = widgets.Dropdown(options=['Left', 'Right', 'None'], value='Left', description='EOG:',
                               style={'description_width': '150px'}, layout=widgets.Layout(width='360px'))
dd_yasa_emg = widgets.Dropdown(options=[YASA_EMG_DECLARED, 'None'], value=YASA_EMG_DECLARED,
                               description='Chin EMG:', style={'description_width': '150px'},
                               layout=widgets.Layout(width='460px'))


def set_emg_options(names):
    """YASA's chin EMG choices after a scan: tool 2's declared EMG (default), none, then the channels of the
    recordings, the EMG-like names first. A rescan keeps the user's choice when still offered."""
    previous = dd_yasa_emg.value
    emg_like = [n for n in names if re.search(r'emg|chin|menton|submental', n, re.IGNORECASE)]
    dd_yasa_emg.options = [YASA_EMG_DECLARED, 'None'] + emg_like + [n for n in names if n not in emg_like]
    dd_yasa_emg.value = previous if previous in dd_yasa_emg.options else YASA_EMG_DECLARED


box_yasa = widgets.VBox(
    [widgets.HTML('<b>YASA</b>'), dd_yasa_eeg, dd_yasa_eog, dd_yasa_emg],
    layout=widgets.Layout(display='' if cb_yasa.value else 'none', margin='6px 0'))

# ---- SleepFM: its model, found on disk or downloaded from the official repository ----
sleepfm_status = widgets.HTML('')
btn_sleepfm_download = widgets.Button(description='Download SleepFM now', icon='download',
                                      layout=widgets.Layout(width='220px'))
sleepfm_progress = widgets.IntProgress(value=0, min=0, max=max(1, SFM.SLEEPFM_TOTAL_BYTES // 1_000_000),
                                       description='Download:', layout=widgets.Layout(width='460px', display='none'))
sleepfm_progress_label = widgets.HTML('')
box_sleepfm = widgets.VBox(
    [widgets.HTML('<b>SleepFM</b> <small style="color:#555;">(EEG derivations, both EOG, chin EMG and ECG '
                  'from tool 2; respiration found by name)</small>'),
     sleepfm_status, btn_sleepfm_download, sleepfm_progress, sleepfm_progress_label],
    layout=widgets.Layout(display='' if cb_sleepfm.value else 'none', margin='6px 0'))


def refresh_sleepfm_status():
    root, how = SFM.find_sleepfm()
    if root is not None:
        sleepfm_status.value = f'<small style="color:#2e7d32;">✅ SleepFM model found: {how}.</small>'
        btn_sleepfm_download.layout.display = 'none'
    else:
        sleepfm_status.value = (f'<small style="color:#e67e22;">SleepFM model {how}: it will be downloaded '
                                f'(~{SFM.SLEEPFM_TOTAL_BYTES / 1e6:.0f} MB, official repository, version '
                                f'{SFM.SLEEPFM_COMMIT[:8]}) when you click Run, or now with the button. '
                                f'Destination: {SFM.cache_root()}</small>')
        btn_sleepfm_download.layout.display = ''
    return root


def ensure_sleepfm(say):
    """The SleepFM folder, downloaded first when needed (progress bar + messages). None if it failed."""
    root, how = SFM.find_sleepfm()
    if root is not None:
        return root
    say(f'Downloading the SleepFM model (~{SFM.SLEEPFM_TOTAL_BYTES / 1e6:.0f} MB) from the official '
        f'repository into {SFM.cache_root()} ...')
    sleepfm_progress.layout.display = ''
    sleepfm_progress.value = 0

    def show(done, total, name):
        sleepfm_progress.value = done // 1_000_000
        sleepfm_progress_label.value = (f'<small>{done / 1e6:.1f} / {total / 1e6:.1f} MB — '
                                        f'{name.split("/")[-1]}</small>')
    try:
        root = SFM.download_sleepfm(progress=show)
        say(f'✅ SleepFM downloaded and checked (every file matches its fingerprint): {root}')
        return root
    except Exception as e:
        say(f'❌ SleepFM download failed: {e}')
        return None
    finally:
        refresh_sleepfm_status()


def _on_download(btn):
    btn.disabled = True
    try:
        ensure_sleepfm(lambda m: setattr(sleepfm_progress_label, 'value', f'<small>{m}</small>'))
    finally:
        btn.disabled = False


btn_sleepfm_download.on_click(_on_download)
refresh_sleepfm_status()


def set_eeg_options(options):
    """Fill the EEG pickers after a scan. GSSC one-electrode mode: C3, else C4, else the first electrode;
    consensus: C3, C4, F3, F4 ticked (those present; else the first electrode).
    YASA: C4 (its preference), else C3, else the first. A rescan keeps the user's choices still valid."""
    previous_one = dd_eeg.value
    previous_ticks = {ch: cb.value for ch, cb in eeg_checks.items()}
    previous_yasa = dd_yasa_eeg.value
    dd_eeg.options = options
    dd_yasa_eeg.options = options
    eeg_checks.clear()
    trained = [ch for ch in options if ch.upper() in GSSC_TRAINED_EEG]
    for ch in options:
        default = ch in trained or (not trained and ch == options[0])
        eeg_checks[ch] = widgets.Checkbox(value=previous_ticks.get(ch, default), description=ch, indent=False,
                                          layout=widgets.Layout(width='auto', margin='2px 16px 2px 0'))
    box_eeg_checks.children = list(eeg_checks.values())
    for cb in eeg_checks.values():
        cb.observe(_update_consensus_hint, names='value')
    _update_consensus_hint()
    if options:
        dd_eeg.value = (previous_one if previous_one in options
                        else next((c for c in ('C3', 'C4') if c in options), options[0]))
        dd_yasa_eeg.value = (previous_yasa if previous_yasa in options
                             else next((c for c in ('C4', 'C3') if c in options), options[0]))
        eeg_hint.value = f'<small style="color:#555;">{len(options)} EEG electrode(s) available.</small>'
    else:
        eeg_hint.value = '<small style="color:#c0392b;">No EEG electrode found in the tool-2 JSON.</small>'


def _on_mode(change):
    one = change['new'].startswith('One')
    dd_eeg.layout.display = '' if one else 'none'
    box_consensus.layout.display = 'none' if one else ''


tb_mode.observe(_on_mode, names='value')


def _on_algorithms(change=None):
    box_gssc.layout.display = '' if cb_gssc.value else 'none'
    box_yasa.layout.display = '' if cb_yasa.value else 'none'
    box_sleepfm.layout.display = '' if cb_sleepfm.value else 'none'
    try:
        refresh_done()              # 'already processed' depends on the models ticked
    except Exception as e:
        scan_info.value += f'<br><span style="color:#c0392b;">{e}</span>'


for _cb in ALGO_CHECKS.values():
    _cb.observe(_on_algorithms, names='value')

display(widgets.HTML('<b>Models:</b>'), widgets.HBox([cb_gssc, cb_yasa, cb_sleepfm]), dd_ref,
        box_gssc, box_yasa, box_sleepfm)

## 3. Run

Scores the selected participants with each ticked model, writes their outputs, then rebuilds each model's
database summary from every per-participant table on disk (so a participant scored in an earlier run is still
included). With *Skip already processed* ticked, a model whose outputs are already on disk for a participant
is not run again.

In [ ]:
btn_run = widgets.Button(description='Run automatic scoring', button_style='success', icon='play',
                         layout=widgets.Layout(width='260px', height='38px'))
progress = widgets.IntProgress(value=0, min=0, max=1, description='Progress:',
                               layout=widgets.Layout(width='460px'))
out_run = widgets.Output()
# Expert switch (no widget): True predicts SleepFM's last minutes with the whole night as context, as
# Score_EDF did (slower); False uses 30 min of context (sleepfm_lib.TAIL_CONTEXT_CHUNKS).
SLEEPFM_FULL_TAIL = False


def log(msg):
    with out_run:
        print(msg)


def expert_path(edf_path, fid):
    """The expert hypnogram of a participant (Section 1 suffix), or None."""
    if not hypno_suffix.value.strip():
        return None
    hp = edf_path.parent / f'{fid}{hypno_suffix.value}'
    if not hp.exists():
        hp = next((p for p in edf_path.parent.iterdir()
                   if os.path.normcase(p.name) == os.path.normcase(hp.name)), hp)
    return hp if hp.exists() else None


def score_model(algo, infer, edf_path, cfg_entry, sleepfm_root):
    """One model on one night: a dict with at least stages, onsets, probs, reference, notes."""
    if algo == 'GSSC':
        one_mode = tb_mode.value.startswith('One')
        wanted = [dd_eeg.value] if one_mode else selected_eeg()
        r = score_night_gssc(infer, edf_path, cfg_entry, wanted, one_mode, dd_eog.value, dd_ref.value)
        r['one_mode'] = one_mode
        return r
    if algo == 'YASA':
        return score_night_yasa(edf_path, cfg_entry, dd_yasa_eeg.value, dd_yasa_eog.value, dd_ref.value,
                                dd_yasa_emg.value)
    device = 'cuda' if (GSSC_OK and torch.cuda.is_available()) else 'cpu'
    r = score_night_sleepfm(edf_path, cfg_entry, sleepfm_root, dd_ref.value, device=device,
                            full_tail=SLEEPFM_FULL_TAIL)
    r['device'] = device
    return r


def score_one(algo, infer, data_root, fid, edf_path, cfg_entry, sleepfm_root=None):
    """Score one participant with one model and write its outputs. Raises on a fatal error."""
    key = ALGO_KEY[algo]
    tag = f'[{fid}]' if algo == 'GSSC' else f'[{fid}] {algo}:'
    r = score_model(algo, infer, edf_path, cfg_entry, sleepfm_root)
    stages, onsets, probs, notes = r['stages'], r['onsets'], r['probs'], r['notes']
    for n in notes:
        log(f'{tag} ⚠ {n}')

    # Expert hypnogram (optional). The models score 30-s epochs from the recording start and drop the
    # incomplete tail, like the scorer's epochs: epoch i = line i of the expert file.
    expert = None
    if hypno_suffix.value.strip():
        hp = expert_path(edf_path, fid)
        if hp is not None:
            expert = load_expert(hp)
            if abs(len(expert) - len(stages)) > 1:
                log(f'{tag} ⚠ length mismatch: expert {len(expert)} epochs vs {algo} {len(stages)}: '
                    'compared on the common start')
        else:
            name = f'{fid}{hypno_suffix.value}'
            notes.append(f'no expert hypnogram ({name}): scored without comparison')
            log(f'{tag} no expert hypnogram ({name}): scored without comparison')

    start_dt = recording_start(edf_path)       # None if unreadable or the EDF+ placeholder 01.01.85 00.00.00
    if start_dt is None:
        notes.append('recording start time unknown in the EDF header (unreadable or anonymised '
                     'placeholder): no clock-time axis')
    paths = output_paths(data_root, edf_path, fid, algo)
    for d in {paths['density'].parent, paths['report'].parent}:
        d.mkdir(parents=True, exist_ok=True)

    # ---- per-participant data first (the report comes last: report on disk => data on disk) ----
    with open(paths['hypno'], 'w', encoding='utf-8') as f:
        f.write('\n'.join(stages) + '\n')
    dens = pd.DataFrame({'file_id': fid, 'epoch_idx': np.arange(len(stages)), 'onset_s': onsets,
                         f'stage_{key}': stages})
    for i, s in enumerate(STAGES):
        dens[f'p_{s}'] = probs[:, i]
    dens['confidence'] = probs.max(axis=1)
    if algo == 'GSSC':
        dens['winning_combination'] = r['winner']      # the signals whose scoring was kept for this epoch
    exp_col = np.full(len(stages), '', dtype=object)
    if expert is not None:
        n = min(len(expert), len(stages))
        exp_col[:n] = expert[:n]
    dens['stage_expert'] = exp_col
    # agree is left empty where the expert label is not W/N1/N2/N3/R (MT, custom, '?', not scored)
    dens['agree'] = np.where(~dens['stage_expert'].isin(STAGES), np.nan,
                             (dens['stage_expert'] == dens[f'stage_{key}']).astype(float))
    dens.to_csv(paths['density'], sep='\t', index=False, float_format='%.4f')
    if algo == 'SleepFM':
        # SleepFM's native 5-s probabilities (full precision: Section 4 averages them over its own epochs)
        # and its internal features + final layer (Section 4's recalibration B).
        five = pd.DataFrame({'file_id': fid, 'step_idx': np.arange(len(r['starts_s'])),
                             'start_s': r['starts_s'], 'stop_s': r['stops_s']})
        for i, s in enumerate(STAGES):
            five[f'p_{s}'] = r['probs_5s'][:, i]
        five['source'] = r['source']
        five.to_csv(paths['density_5s'], sep='\t', index=False)
        np.savez_compressed(paths['features'], features=r['features'], fc_weight=r['fc_weight'],
                            fc_bias=r['fc_bias'], start_s=r['starts_s'])

    common_end = {'expert_hypnogram_suffix': hypno_suffix.value if expert is not None else None,
                  'recording_start': start_dt.isoformat() if start_dt else None,
                  'notes': notes, 'scored_at': datetime.datetime.now().isoformat(timespec='seconds')}
    if algo == 'GSSC':
        eog_used = ', '.join(r['eog_names']) if r['eog_names'] else 'none'
        params = {'file_id': fid, 'scoring_mode': 'one_electrode' if r['one_mode'] else 'consensus',
                  'consensus_rule': CONSENSUS_RULE, 'n_combinations': r['n_comb'],
                  'hypnodensity': 'softmax of the winning combination (gssc 0.0.9 loop, logits kept)',
                  'eeg_used': r['present'], 'eog_used': r['eog_names'], 'eog_mode': dd_eog.value,
                  'reference': cfg_entry.get('ref_channels', []), 'eeg_reference': r['reference'],
                  'eog_declared': cfg_entry.get('context_channels') or {},
                  'expert_hypnogram_suffix': common_end['expert_hypnogram_suffix'],
                  'gssc_version': GSSC_VERSION, 'torch_version': TORCH_VERSION,
                  'device': 'cuda' if torch.cuda.is_available() else 'cpu',
                  'recording_start': common_end['recording_start'],
                  'notes': notes, 'scored_at': common_end['scored_at']}
        details = {'scoring_mode': params['scoring_mode'], 'eeg_used': ', '.join(r['present']),
                   'eog_used': eog_used}
        shown = {'EEG used': ', '.join(r['present']), 'EOG used': eog_used,
                 'Mode': f'{params["scoring_mode"]} ({r["n_comb"]} signal combinations, rule: most confident per epoch)',
                 'Reference (tool 2)': params['reference'], 'EEG reference (model)': r['reference'],
                 'GSSC / torch': f'{GSSC_VERSION} / {TORCH_VERSION}'}
    elif algo == 'YASA':
        used = r['used']
        params = {'file_id': fid, 'algorithm': 'YASA', 'eeg_used': used['eeg'], 'eog_used': used['eog'],
                  'emg_used': used['emg'], 'eog_choice': dd_yasa_eog.value, 'emg_choice': dd_yasa_emg.value,
                  'eeg_reference': r['reference'],
                  'reference': cfg_entry.get('ref_channels', []),
                  'context_declared': cfg_entry.get('context_channels') or {},
                  'demographics': 'not used (classifier without age/sex)', 'yasa_version': YASA_VERSION,
                  **common_end}
        details = {'eeg_used': used['eeg'], 'eog_used': used['eog'] or 'none', 'emg_used': used['emg'] or 'none'}
        shown = {'EEG used': used['eeg'], 'EOG used': used['eog'] or 'none', 'Chin EMG used': used['emg'] or 'none',
                 'EEG reference (model)': r['reference'], 'YASA': YASA_VERSION}
    else:
        used = r['used']
        params = {'file_id': fid, 'algorithm': 'SleepFM-Clinical', 'channels': used,
                  'eeg_reference': r['reference'], 'reference': cfg_entry.get('ref_channels', []),
                  'context_declared': cfg_entry.get('context_channels') or {},
                  'sleepfm_commit': SFM.SLEEPFM_COMMIT, 'sleepfm_folder': str(sleepfm_root),
                  'native_resolution_s': SFM.STEP_S,
                  'tail_context_min': None if r['tail_context_chunks'] is None else r['tail_context_chunks'] * 5,
                  'bandpass_hz': SFM.BANDPASS, 'device': r['device'], 'torch_version': TORCH_VERSION,
                  **common_end}
        details = {'eeg_used': ', '.join(used['BAS']), 'emg_used': ', '.join(used['EMG']) or 'none',
                   'resp_used': ', '.join(used['RESP']) or 'none'}
        shown = {'Brain channels (EEG + EOG)': ', '.join(used['BAS']), 'Chin EMG': ', '.join(used['EMG']) or 'none',
                 'ECG': ', '.join(used['EKG']) or 'none', 'Respiration': ', '.join(used['RESP']) or 'none',
                 'EEG reference (model)': r['reference'],
                 'SleepFM version': f'{SFM.SLEEPFM_COMMIT[:8]} (torch {TORCH_VERSION}, {r["device"]})'}
    with open(paths['params'], 'w', encoding='utf-8') as f:
        json.dump(params, f, indent=2, ensure_ascii=False, default=str)

    if expert is not None:
        res, cm, cm_long = compare_scoring(expert, stages, key)
        status = 'compared' if res['n_compared'] else 'no expert W/N1/N2/N3/R epoch'
    else:
        res, cm, cm_long = None, None, None
        status = 'no expert hypnogram'
    row = {'file_id': fid, f'n_epochs_{key}': len(stages),
           'n_epochs_expert': len(expert) if expert is not None else np.nan,
           'n_compared': res['n_compared'] if res else 0,
           'n_excluded_expert': res['n_excluded_expert'] if res else np.nan,
           'accuracy': res['accuracy'] if res else np.nan, 'kappa': res['kappa'] if res else np.nan,
           **{f'f1_{s}': (res[f'f1_{s}'] if res else np.nan) for s in STAGES},
           'mean_confidence': float(probs.max(axis=1).mean()),
           **details, 'status': status}
    pd.DataFrame([row]).to_csv(paths['comparison'], sep='\t', index=False, float_format='%.4f')
    # Expert epochs outside W/N1/N2/N3/R: what the model makes of them (only when the night has some).
    custom_path = paths['comparison'].parent / f'{fid}_custom_stages.tsv'
    profile = (non_aasm_profile(expert, stages, probs, STATE['custom'], key) if expert is not None
               else pd.DataFrame())
    if len(profile):
        profile.insert(0, 'file_id', fid)
        profile.to_csv(custom_path, sep='\t', index=False, float_format='%.4f')
    elif custom_path.exists():
        custom_path.unlink()                 # a former profile no longer valid
    if cm_long is not None:
        cm_long.insert(0, 'file_id', fid)
        cm_long.to_csv(paths['confusion'], sep='\t', index=False)
    elif paths['confusion'].exists():
        paths['confusion'].unlink()          # a former comparison no longer valid

    # ---- report ----
    shown.update({'Recording start (EDF header)': f'{start_dt:%Y-%m-%d %H:%M:%S}' if start_dt else 'unreadable',
                  'Epochs scored': len(stages), 'Mean confidence': f'{row["mean_confidence"]:.2f}'})
    if notes:
        shown['Notes'] = '<br>'.join(f'⚠ {n}' for n in notes)
    items = [('html', 'Scoring parameters', params_html(shown)),
             ('fig', 'Hypnograms', plot_hypnograms(stages, expert, fid, STATE['custom'], start_dt, algo)),
             ('fig', 'Hypnodensity', plot_hypnodensity(probs, fid, expert, start_dt))]
    if res is not None and res['n_compared']:
        items += [('html', 'Agreement with the expert', metrics_html(res)),
                  ('fig', 'Confusion matrix', plot_confusion(
                      cm, f'{fid}: kappa {res["kappa"]:.2f}, accuracy {res["accuracy"] * 100:.0f} %', algo))]
    else:
        items += [('html', 'Agreement with the expert',
                   f'<p>No comparison: {status}.</p>')]
    if len(profile):
        items += [('html', 'Expert stages outside W/N1/N2/N3/R', non_aasm_help(algo)),
                  ('fig', f'Expert stages outside W/N1/N2/N3/R: what {algo} makes of them',
                   plot_non_aasm_profile(profile, f'{fid}: expert stages outside W/N1/N2/N3/R, '
                                                  'with the AASM stages as reference', algo, key))]
    save_report_html(paths['report'], f'Automatic sleep scoring ({algo}): {fid}', items)
    return row


def night_start_hours(data_root, density_path, fid):
    """Recording start of a scored night on the noon-to-noon clock scale, or None: from its
    _scoring_params.json (recording_start), else from the EDF header (nights scored before that field)."""
    start_dt = None
    try:
        with open(density_path.parent / f'{fid}_scoring_params.json', encoding='utf-8') as f:
            iso = json.load(f).get('recording_start')
        start_dt = datetime.datetime.fromisoformat(iso) if iso else None
    except Exception:
        start_dt = None
    if start_dt is None:
        edf = STATE['edf_by_id'].get(fid) or next(
            (e for e in data_root.rglob('*') if e.suffix.lower() == '.edf'
             and os.path.normcase(e.stem) == os.path.normcase(fid)), None)
        start_dt = recording_start(edf) if edf else None
    return clock_hours(start_dt) if start_dt else None


def rebuild_database_summary(data_root, algo='GSSC'):
    """One model's database tables + report, rebuilt from every per-participant table on disk."""
    key = ALGO_KEY[algo]
    rep_root = data_root / f'reports_autoscoring_{key}'
    if not rep_root.exists():
        return
    # Per-participant tables only: the database tables written here (global_*) share their suffixes
    # (global_confusion.tsv, global_custom_stages.tsv) and would otherwise be counted again.
    def per_file(pattern):
        return sorted(p for p in rep_root.rglob(pattern) if not p.name.startswith('global_'))

    comps = per_file('*_autoscoring_comparison.tsv')
    if not comps:
        return
    summ = pd.concat([pd.read_csv(p, sep='\t', dtype={'file_id': str}) for p in comps], ignore_index=True)
    summ = summ.sort_values('file_id')
    summ.to_csv(rep_root / 'global_autoscoring_summary.tsv', sep='\t', index=False, float_format='%.4f')
    confs = per_file('*_confusion.tsv')
    items = []
    if confs:
        allc = pd.concat([pd.read_csv(p, sep='\t', dtype={'file_id': str}) for p in confs], ignore_index=True)
        pooled = allc.groupby(['expert', key], as_index=False)['n'].sum()
        pooled.to_csv(rep_root / 'global_confusion.tsv', sep='\t', index=False)
        cm = (pooled.pivot(index='expert', columns=key, values='n')
              .reindex(index=STAGES, columns=STAGES).fillna(0).astype(int).values)
        n_tot = cm.sum()
        acc = np.trace(cm) / n_tot if n_tot else np.nan
        items.append(('fig', 'Pooled confusion matrix', plot_confusion(
            cm, f'All participants pooled ({allc["file_id"].nunique()} nights, {n_tot} epochs), '
                f'accuracy {acc * 100:.0f} %', algo)))
    customs = per_file('*_custom_stages.tsv')
    if customs:
        allp = pd.concat([pd.read_csv(p, sep='\t', dtype={'file_id': str}) for p in customs], ignore_index=True)
        pooled_p = pool_profiles(allp, STATE['custom'], key)
        pooled_p.to_csv(rep_root / 'global_custom_stages.tsv', sep='\t', index=False, float_format='%.4f')
        items.append(('html', 'Expert stages outside W/N1/N2/N3/R',
                      non_aasm_help(algo) + f'<p style="font-size:85%;color:#555">Pooled over the '
                      f'{allp["file_id"].nunique()} night(s) that have such epochs (reference bars from the '
                      'same nights).</p>'))
        items.append(('fig', f'Expert stages outside W/N1/N2/N3/R: what {algo} makes of them',
                      plot_non_aasm_profile(pooled_p, 'All nights pooled: expert stages outside W/N1/N2/N3/R',
                                            algo, key)))
    elif (rep_root / 'global_custom_stages.tsv').exists():
        (rep_root / 'global_custom_stages.tsv').unlink()
    comp = summ[summ['n_compared'] > 0]
    if len(comp):
        fig, ax = plt.subplots(figsize=(7, 3.2))
        for i, col in enumerate(['kappa', 'accuracy']):
            vals = comp[col].astype(float).values
            ax.scatter(np.full(len(vals), i) + np.random.default_rng(0).uniform(-0.12, 0.12, len(vals)),
                       vals, s=18, color='#3a87c8', alpha=0.8)
            ax.plot([i - 0.25, i + 0.25], [np.nanmedian(vals)] * 2, color='#222222', lw=2)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["Cohen's kappa", 'Accuracy'])
        ax.set_ylim(0, 1)
        ax.set_title('Agreement with the expert per participant (line = median)', fontsize=10)
        ax.grid(axis='y', alpha=0.3)
        fig.tight_layout()
        items.append(('fig', 'Agreement per participant', fig))
    # Every participant's hypnodensity stacked on one time axis, for a quick visual comparison.
    dens_files = sorted((data_root / 'derivatives' / f'autoscoring_{key}').rglob('*_hypnodensity.tsv'),
                        key=lambda p: p.name)
    kappa_by_id = dict(zip(summ['file_id'].astype(str), pd.to_numeric(summ['kappa'], errors='coerce')))
    nights = []
    for p in dens_files:
        try:
            d = pd.read_csv(p, sep='\t', dtype={'file_id': str}, keep_default_na=False)
            fid = str(d['file_id'].iloc[0])
            k = kappa_by_id.get(fid, np.nan)
            label = fid if pd.isna(k) else f'{fid}\nκ {k:.2f}'
            expert = d['stage_expert'].astype(str).values if 'stage_expert' in d else None
            if expert is not None and not (expert != '').any():
                expert = None                                # scored without an expert hypnogram
            nights.append((label, d[[f'p_{s}' for s in STAGES]].astype(float).values, expert,
                           night_start_hours(data_root, p, fid)))
        except Exception as e:
            log(f'⚠ Hypnodensity not shown in the database report ({p.name}): {e}')
    # Nights with a known start time on the real clock (one shared axis for every figure); the others
    # (start unreadable in the EDF header) apart, on the relative axis.
    clocked = [n for n in nights if n[3] is not None]
    unclocked = [n for n in nights if n[3] is None]
    n_per = HYPNODENSITY_ROWS_PER_FIG
    if clocked:
        x_rng = (np.floor(min(n[3] for n in clocked)),
                 np.ceil(max(n[3] + len(n[1]) * EPOCH_SEC / 3600 for n in clocked)))
        for start in range(0, len(clocked), n_per):
            part = clocked[start:start + n_per]
            title = (f'Hypnodensities on clock time, participants {start + 1}-{start + len(part)} of '
                     f'{len(clocked)}' if len(clocked) > n_per
                     else f'Hypnodensities on clock time ({len(clocked)} participants)')
            items.append(('fig', title, plot_hypnodensity_stack(part, x_rng, title, clock=True)))
    if unclocked:
        x_rng = (0, max(len(n[1]) for n in unclocked) * EPOCH_SEC / 3600)
        for start in range(0, len(unclocked), n_per):
            part = unclocked[start:start + n_per]
            title = (f'Hypnodensities, recording start unknown in the EDF header ({len(unclocked)} '
                     f'participant(s), relative time)')
            items.append(('fig', title, plot_hypnodensity_stack(part, x_rng, title)))
    cols = ['file_id', f'n_epochs_{key}', 'n_compared', 'accuracy', 'kappa', 'mean_confidence',
            'scoring_mode', 'eeg_used', 'eog_used', 'emg_used', 'resp_used', 'status']
    items.append(('html', 'Participants', summ[[c for c in cols if c in summ]].to_html(
        index=False, float_format=lambda v: f'{v:.3f}', na_rep='–')))
    save_report_html(rep_root / 'global_autoscoring_report.html', f'Automatic sleep scoring ({algo}): database',
                     items)


def update_failed(data_root, algo, failed, attempted):
    """failed_autoscoring.tsv of one model: this run's failures + the earlier ones of participants not retried."""
    rep_root = data_root / f'reports_autoscoring_{ALGO_KEY[algo]}'
    fpath = rep_root / 'failed_autoscoring.tsv'
    old = pd.read_csv(fpath, sep='\t', dtype={'file_id': str}) if fpath.exists() else pd.DataFrame()
    if len(old):
        tried = {os.path.normcase(f) for f in attempted}
        old = old[~old['file_id'].map(os.path.normcase).isin(tried)]
    allf = pd.concat([old, pd.DataFrame(failed)], ignore_index=True)
    if len(allf):
        rep_root.mkdir(parents=True, exist_ok=True)
        allf.to_csv(fpath, sep='\t', index=False)
    elif fpath.exists():
        fpath.unlink()


def run_scoring(btn):
    btn.disabled = True
    out_run.clear_output()
    try:
        algos = ticked_algorithms()
        if not algos:
            log('⛔ Tick at least one model (Section 2).')
            return
        if 'GSSC' in algos and not GSSC_OK:
            with out_run:
                display(HTML(GSSC_MSG))
            return
        if not fc_data.selected or not STATE['cfg']:
            log('Select the study folder and click Scan (Section 1) first.')
            return
        suffixes = {a: OUT_SUFFIX[a].value.strip() for a in algos}
        for a, suf in suffixes.items():
            if hypno_suffix.value.strip() and os.path.normcase(hypno_suffix.value.strip()) == os.path.normcase(suf):
                log(f'⛔ The {a} output suffix equals the expert hypnogram suffix: the expert hypnograms would be '
                    'overwritten. Change the output suffix.')
                return
            if not suf.lower().endswith('.txt'):
                log(f'⛔ The {a} output suffix must end with .txt.')
                return
        if len({os.path.normcase(s) for s in suffixes.values()}) < len(suffixes):
            log('⛔ Two models have the same output suffix: give each its own.')
            return
        one_mode = tb_mode.value.startswith('One')
        wanted = [dd_eeg.value] if one_mode else selected_eeg()
        if 'GSSC' in algos and (not wanted or wanted == [None]):
            log('⛔ Choose at least one EEG electrode for GSSC (Section 2).')
            return
        if 'YASA' in algos and not dd_yasa_eeg.value:
            log('⛔ Choose the EEG electrode of YASA (Section 2).')
            return
        sleepfm_root = None
        if 'SleepFM' in algos:
            sleepfm_root = ensure_sleepfm(log)
            if sleepfm_root is None:
                log('⚠ SleepFM is left out of this run (model unavailable); the other models still run.')
                algos = [a for a in algos if a != 'SleepFM']
                if not algos:
                    return
        data_root = Path(fc_data.selected)
        selected = participant_selector.selected_ids()
        if participant_selector.warning_html():
            with out_run:
                display(HTML(participant_selector.warning_html()))
        not_cfg = [f for f in selected if f in STATE['not_in_cfg']]
        if not_cfg:
            log(f'⚠ Not in the tool-2 JSON, not scored: {", ".join(not_cfg)}')
        selected = [f for f in selected if f in STATE['cfg']]
        log(f'{len(selected)} participant(s) to score with {", ".join(algos)}; EEG reference: {dd_ref.value}.')
        if 'GSSC' in algos:
            log(f'GSSC: EEG {", ".join(wanted)}, EOG {dd_eog.value}.')
        if 'YASA' in algos:
            log(f'YASA: EEG {dd_yasa_eeg.value}, EOG {dd_yasa_eog.value}, chin EMG: {dd_yasa_emg.value}.')

        failed = {a: [] for a in algos}
        attempted = {a: set() for a in algos}
        progress.max = max(len(selected) * len(algos), 1)     # one step per participant and model
        progress.value = 0
        infer = None
        if selected and 'GSSC' in algos:
            try:
                infer = EEGInfer(use_cuda=torch.cuda.is_available())
            except Exception as e:
                log(f'⛔ Cannot load the GSSC model: {e}')
                return
        for i, fid in enumerate(selected):
            edf = STATE['edf_by_id'][fid]
            for algo in algos:
                # Skip ticked: a model whose outputs are complete is not run again (the others still are).
                if participant_selector.cb_skip.value and algo_outputs(data_root, edf, fid, algo)[0]:
                    log(f'[{fid}] {algo}: already scored, skipped.')
                    progress.value += 1
                    continue
                attempted[algo].add(fid)
                t0 = time.time()
                try:
                    row = score_one(algo, infer, data_root, fid, edf, STATE['cfg'][fid], sleepfm_root)
                    kappa = '' if pd.isna(row['kappa']) else f', kappa {row["kappa"]:.2f}'
                    log(f'[{fid}] ✅ {algo}: {row[f"n_epochs_{ALGO_KEY[algo]}"]} epochs scored{kappa} '
                        f'({time.time() - t0:.0f} s)')
                except Exception as e:
                    log(f'[{fid}] ❌ {algo}: {e}')
                    failed[algo].append({'file_id': fid, 'reason': str(e)})
                plt.close('all')
                progress.value += 1

        for algo in algos:
            try:
                update_failed(data_root, algo, failed[algo], attempted[algo])
            except Exception as e:
                log(f'⚠ Could not update {algo} failed_autoscoring.tsv: {e}')
            try:
                rebuild_database_summary(data_root, algo)
                log(f'{algo} database summary rebuilt: {data_root / ("reports_autoscoring_" + ALGO_KEY[algo])}')
            except Exception as e:
                log(f'⚠ Could not rebuild the {algo} database summary: {e}')
        n_fail = sum(len(f) for f in failed.values())
        n_runs = sum(len(a) for a in attempted.values())
        log(f'Done: {n_runs - n_fail} scoring(s) done, {n_fail} failed.')
        refresh_done()
    except Exception as e:
        log(f'❌ Unexpected error: {e}')
    finally:
        btn.disabled = False


btn_run.on_click(run_scoring)
display(btn_run, progress, out_run)

## 4. Compare the models

Compares the models scored in Section 3 with the expert hypnogram, on every participant chosen in Section 1
(the *Skip* option does not apply here) that has an expert hypnogram. Only the files written by Section 3 are
read: nothing is scored again. Click **Find the scorings**, choose the models to compare, then **Compare**.

How it works (the computation of Score_EDF): 30-s epochs from the first to the last scored expert epoch; each
model's probabilities averaged over each epoch; the predicted stage is the most probable one. SleepFM's two
**recalibrations** are learnt on the expert hypnograms of the other nights (leave one night out): A needs at
least 2 nights with SleepFM, B at least 3 (6 or more advised).

**Outputs** in `{data_folder}/autoscore_outputs/` (the files of Score_EDF): `epochs_all_stagers.csv` (one row
per epoch and model), `agreement_summary.csv`, and the figures below as PNG; plus `channels_used.csv`, the
channels each model used on each night (each model is given the channels it was trained on, so two models
may not see the same EEG: to compare them on equal inputs, give them the same electrode in Section 2).

**How to read the results**
- **Agreement table**: per model, the number of epochs compared, the **accuracy** (share of epochs where the
  model gives the expert's stage), **Cohen's kappa** (agreement corrected for chance: 0.61-0.80 good, above
  0.80 excellent; two human scorers reach about 0.76, i.e. ~83 % of identical epochs), the **F1 per stage**
  (balance between finding every epoch of the stage and not inventing any; N1 is always the lowest, between
  humans too) and the kappa of each night.
- **Agreement figure**: left, F1 per stage and model; right, the kappa of each night (one symbol per night, the
  black dash = all nights pooled). A night far below the others for every model points to a signal or scoring
  problem rather than to a model.
- **Confusion matrices**: rows = expert stage, columns = predicted stage, % of the row. The diagonal = agreement.
  Usual confusions: N1 with wake, N1 with N2, N2 with N3.
- **Hypnogram grid**: one band per night, the expert on top then one line per model, one colour per stage (grey
  = no data): where the models depart from the expert across the night, at a glance.
- **Hypnodensities**: per night, each model's probability of each stage across the night (stacked areas), the
  expert hypnogram below. One dominant colour = a confident model; mixed colours = an ambiguous epoch (often
  transitions and N1). They show the confidence, not only the stage.

In [ ]:
cmp_checks = {name: widgets.Checkbox(value=False, description=name, disabled=True, indent=False,
                                     layout=widgets.Layout(width='420px')) for name in ACL.STAGERS}
btn_compare_find = widgets.Button(description='Find the scorings', button_style='info', icon='search',
                                  layout=widgets.Layout(width='200px', height='34px'))
cmp_info = widgets.HTML('')
btn_compare = widgets.Button(description='Compare the models', button_style='success', icon='play',
                             layout=widgets.Layout(width='260px', height='38px'))
out_compare = widgets.Output()
out_results = widgets.Output()
# Expert switch (no widget): False reproduces Score_EDF's recalibrations exactly (N3 never fitted); 'auto'
# fits N3 when the training nights have N3 (plan 2.7).
RECAL_FREE_N3 = 'auto'
COMPARE = {}


def clog(msg):
    with out_compare:
        print(msg)


def step1_ready(data_root, edf, fid, algo):
    p = output_paths(data_root, edf, fid, algo)
    return p['density'].exists() and (algo != 'SleepFM' or p['density_5s'].exists())


def step1_start(data_root, edf, fid):
    """Recording start read from step 1's parameter files (no EDF read), else None."""
    for algo in ALGORITHMS:
        p = output_paths(data_root, edf, fid, algo)['params']
        try:
            iso = json.loads(p.read_text(encoding='utf-8')).get('recording_start')
            if iso:
                return datetime.datetime.fromisoformat(iso)
        except Exception:
            continue
    return None


def compare_candidates(data_root):
    """Participants chosen in Section 1 that have an expert hypnogram, with the models scored for them."""
    rows = {}
    for fid in participant_selector.chosen_ids():
        edf = STATE['edf_by_id'].get(fid)
        if edf is None:
            continue
        hp = expert_path(edf, fid)
        if hp is None:
            continue
        rows[fid] = {'edf': edf, 'expert': hp,
                     'algos': [a for a in ALGORITHMS if step1_ready(data_root, edf, fid, a)]}
    return rows


def on_compare_find(btn):
    btn.disabled = True
    try:
        if not fc_data.selected or not STATE['edf_by_id']:
            cmp_info.value = '<span style="color:#c0392b;">Scan the study folder first (Section 1).</span>'
            return
        data_root = Path(fc_data.selected)
        cands = compare_candidates(data_root)
        n = {a: sum(a in c['algos'] for c in cands.values()) for a in ALGORITHMS}
        n_feat = sum('SleepFM' in c['algos'] and output_paths(data_root, c['edf'], fid, 'SleepFM')['features'].exists()
                     for fid, c in cands.items())
        notes = {a: (f' ({n[a]} night(s))' if n[a] else ' (not scored in Section 3)') for a in ALGORITHMS}
        enabled = {a: n[a] > 0 for a in ALGORITHMS}
        enabled['SleepFM recal A'] = n['SleepFM'] >= 2
        notes['SleepFM recal A'] = (f' (learnt on the other nights: {n["SleepFM"]} with SleepFM)' if n['SleepFM'] >= 2
                                    else f' (needs >= 2 nights with SleepFM, have {n["SleepFM"]})')
        enabled['SleepFM recal B'] = n_feat >= 3
        notes['SleepFM recal B'] = (f' ({n_feat} nights; works best with >= 6)' if 3 <= n_feat < 6
                                    else (f' ({n_feat} nights)' if n_feat >= 6
                                          else f' (needs >= 3 nights with SleepFM, have {n_feat})'))
        for name, cb in cmp_checks.items():
            cb.disabled = not enabled[name]
            cb.value = enabled[name]
            cb.description = name + notes[name]
        cmp_info.value = (f'<span style="color:#2e7d32;">{len(cands)} participant(s) with an expert hypnogram '
                          f'({hypno_suffix.value}).</span>' if cands else
                          '<span style="color:#c0392b;">No participant with an expert hypnogram (check the expert '
                          'suffix in Section 1).</span>')
        COMPARE['candidates'] = cands
    except Exception as e:
        cmp_info.value = f'<span style="color:#c0392b;">Error: {e}</span>'
    finally:
        btn.disabled = False


def channels_used(data_root, cands, selected):
    """One row per night and model: the channels and the EEG reference the model used, read from step 1's
    _scoring_params.json (no EDF read)."""
    models = [a for a in ALGORITHMS if a in selected or (a == 'SleepFM' and any(s.startswith('SleepFM')
                                                                                 for s in selected))]
    rows = []
    for fid, c in cands.items():
        for algo in models:
            if algo not in c['algos']:
                continue
            try:
                d = json.loads(output_paths(data_root, c['edf'], fid, algo)['params'].read_text(encoding='utf-8'))
            except Exception:
                continue
            if algo == 'GSSC':
                # GSSC records its EOG as EOG-L / EOG-R / EOG L-R: shown with the channels declared in tool 2
                decl = d.get('eog_declared') or {}
                names = {'EOG-L': decl.get('eog_left'), 'EOG-R': decl.get('eog_right'),
                         'EOG L-R': f"{decl.get('eog_left')} - {decl.get('eog_right')}"}
                eog = ', '.join(names.get(e) or e for e in (d.get('eog_used') or [])) or 'none'
                row = {'EEG': ', '.join(d.get('eeg_used') or []), 'EOG': eog,
                       'chin EMG': '-', 'other': f"{d.get('scoring_mode')} ({d.get('n_combinations')} combinations)"}
            elif algo == 'YASA':
                row = {'EEG': d.get('eeg_used'), 'EOG': d.get('eog_used') or 'none',
                       'chin EMG': d.get('emg_used') or 'none', 'other': ''}
            else:
                ch = d.get('channels') or {}
                eeg = [n for n in ch.get('BAS', []) if not n.startswith('EOG')]
                row = {'EEG': ', '.join(eeg), 'EOG': ', '.join(ch.get('EOG', [])) or 'none',
                       'chin EMG': ', '.join(ch.get('EMG', [])) or 'none',
                       'other': f"ECG: {', '.join(ch.get('EKG', [])) or 'none'}; respiration: "
                                f"{', '.join(ch.get('RESP', [])) or 'none'}"}
            rows.append({'subject': fid, 'model': algo, **row, 'EEG reference': d.get('eeg_reference', '')})
    return pd.DataFrame(rows)


def run_compare(btn):
    btn.disabled = True
    out_compare.clear_output()
    out_results.clear_output()
    try:
        data_root = Path(fc_data.selected) if fc_data.selected else None
        cands = COMPARE.get('candidates')
        selected = [n for n, cb in cmp_checks.items() if cb.value and not cb.disabled]
        if data_root is None or not cands:
            clog('Click "Find the scorings" first.')
            return
        if not selected:
            clog('⛔ Tick at least one model to compare.')
            return
        out_dir = data_root / 'autoscore_outputs'
        out_dir.mkdir(parents=True, exist_ok=True)
        clog(f'Output directory: {out_dir}')
        clog(f'Models compared: {selected}')
        clog(f'Recordings: {len(cands)}')
        needs_sleepfm = any(s.startswith('SleepFM') for s in selected)
        five = {name: {} for name in selected}
        native = {}
        info = {}
        failed = []
        for fid, c in cands.items():
            try:
                hyp = ACL.read_expert(c['expert'])
                window = ACL.window_from_expert(hyp, step1_start(data_root, c['edf'], fid))
                clog(f'--- {fid} ---  expert: {c["expert"].name} ({len(hyp)} epochs), window '
                     f'{window.lights_off_clock} -- {window.reveil_clock} ({window.duration_seconds / 60:.1f} min)')
                info[fid] = {'window': window, 'hypnogram': hyp}
                for algo in ('YASA', 'GSSC', 'SleepFM'):
                    if algo not in c['algos'] or not (algo in selected or (algo == 'SleepFM' and needs_sleepfm)):
                        continue
                    p = output_paths(data_root, c['edf'], fid, algo)
                    frame, nat = ACL.step1_frame(p['density'], p['density_5s'] if algo == 'SleepFM' else None)
                    if algo in selected:
                        five[algo][fid] = frame
                        native[(algo, fid)] = nat
                    if algo == 'SleepFM':
                        info[fid]['sleepfm_5s'] = frame
                        info[fid]['features'] = p['features']
            except Exception as e:
                clog(f'[{fid}] ❌ {e}')
                failed.append((fid, str(e)))

        # ---- SleepFM recalibrations, leave one recording out ----
        recal_subjects = [fid for fid in info if 'sleepfm_5s' in info[fid]]
        needs_recal = any(s in selected for s in ('SleepFM recal A', 'SleepFM recal B'))
        if needs_recal and len(recal_subjects) >= 2:
            base_recs = [ACL.prepare_recording(fid, info[fid]['sleepfm_5s'], info[fid]['window'],
                                               info[fid]['hypnogram']) for fid in recal_subjects]
            if 'SleepFM recal A' in selected:
                clog('\n--- Recalibration A (leave one recording out) ---')
                five['SleepFM recal A'] = ACL.recalibrate_a(base_recs, RECAL_FREE_N3, clog)
            if 'SleepFM recal B' in selected and len(base_recs) >= 3:
                clog('\n--- Recalibration B (leave one recording out, refit of the final layer) ---')
                feats, fc_w, fc_b = {}, None, None
                for fid in recal_subjects:
                    d = np.load(info[fid]['features'])
                    feats[fid], fc_w, fc_b = d['features'], d['fc_weight'], d['fc_bias']
                five['SleepFM recal B'] = ACL.recalibrate_b(base_recs, feats, fc_w, fc_b, RECAL_FREE_N3, clog)

        # ---- one table of epochs ----
        clog('\nBuilding epoch tables...')
        active = [s for s in selected if five.get(s)]
        frames = []
        for name in active:
            for fid, frame in five[name].items():
                w = info[fid]['window']
                table = ACL.build_epoch_table(w, info[fid]['hypnogram'], frame, 30.0,
                                              native_stages=native.get((name, fid)))
                table['minutes_from_start'] = (table['start_seconds'] - w.lights_off_seconds) / 60
                frames.append(table.assign(subject=fid, stager=name))
        if not frames:
            clog('No results to display.')
            return
        epochs_df = pd.concat(frames, ignore_index=True)
        scored_df = epochs_df.loc[epochs_df['manual_stage'] != 'Unknown']
        subjects = sorted(info)
        epochs_df.to_csv(out_dir / 'epochs_all_stagers.csv', index=False)
        clog(f'Saved epoch table to {out_dir / "epochs_all_stagers.csv"}')
        if failed:
            clog(f'\nFailed recordings ({len(failed)}):')
            for fid, err in failed:
                clog(f'  {fid}: {err}')
        COMPARE.update({'epochs_df': epochs_df, 'scored_df': scored_df, 'active': active, 'subjects': subjects,
                        'out_dir': out_dir, 'channels': channels_used(data_root, cands, selected)})
        show_comparison()
        clog('Done.')
    except Exception as e:
        clog(f'❌ Unexpected error: {e}')
    finally:
        btn.disabled = False


def show_comparison():
    """The results, saved as Score_EDF saved them and shown below."""
    epochs_df, scored_df = COMPARE['epochs_df'], COMPARE['scored_df']
    stagers, subjects, out_dir = COMPARE['active'], COMPARE['subjects'], COMPARE['out_dir']
    with out_results:
        display(HTML('<h3>4.1 Agreement with the expert hypnogram</h3>'))
        agreement = ACL.compute_agreement(scored_df, stagers, subjects)
        agreement.to_csv(out_dir / 'agreement_summary.csv')
        display(agreement.round(3).style.background_gradient(cmap='Greens', subset=['accuracy', 'kappa_pooled']))
        channels = COMPARE.get('channels')
        if channels is not None and len(channels):
            channels.to_csv(out_dir / 'channels_used.csv', index=False)
            display(HTML('<h4>Channels used by each model</h4><p style="font-size:85%;color:#555">Each model '
                         'is given the channels it was trained on: a difference between two models can come '
                         'from their channels (e.g. a poor electrode) as much as from the models.</p>'))
            display(channels)
        figures = [('agreement_by_stager', '', lambda: ACL.plot_agreement_summary(agreement, stagers, subjects)),
                   ('confusion_by_stager', '<h3>4.2 Confusion matrices</h3>',
                    lambda: ACL.plot_confusion_matrices(scored_df, stagers)),
                   ('hypnograms_by_stager', '<h3>4.3 Hypnogram comparison</h3>',
                    lambda: ACL.plot_hypnogram_grid(epochs_df, stagers, subjects))]
        figures += [(f'hypnodensities_{sid}', '<h3>4.4 Hypnodensities per recording</h3>' if i == 0 else '',
                     (lambda sid=sid: ACL.plot_hypnodensities(epochs_df, stagers, sid)))
                    for i, sid in enumerate(subjects)]
        for name, heading, make in figures:
            if heading:
                display(HTML(heading))
            try:
                fig = make()
                png = out_dir / f'{name}.png'
                fig.savefig(png, dpi=130, bbox_inches='tight')
                plt.close(fig)
                # This tool draws without a screen (Agg backend, cell 1): display(fig) would print only
                # '<Figure size ...>'. Show the PNG just saved, as tools 0, 8 and 8bis do.
                display(widgets.Image(value=png.read_bytes(), format='png',
                                      layout=widgets.Layout(max_width='100%', height='auto')))
            except Exception as e:
                print(f'{name}: plot error: {e}')
        display(HTML(f'<hr><p>All outputs saved to <code>{out_dir}</code></p>'))


btn_compare_find.on_click(on_compare_find)
btn_compare.on_click(run_compare)
display(btn_compare_find, cmp_info, widgets.HTML('<b>Models to compare:</b>'),
        widgets.VBox(list(cmp_checks.values())), btn_compare, out_compare, out_results)